In [446]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [447]:
import sys

sys.path.append("..")

In [448]:

from constrerl.utils import extract_flags_from_name
import glob
from pathlib import Path
import json
import pandas as pd
from collections.abc import Callable, Awaitable


In [449]:
test_file = Path("./results_test/all.csv")
test_df = pd.read_csv(test_file)
report_dir = Path("report")

test_df = test_df[test_df["Team ID"] == "TUGW"]
report_dir = report_dir / "test-tugw"
lbl_xtra = ":test:tugw"
mode = "Merged Test"
# test_df = test_df[test_df["Team ID"] == "ToGS"]
# report_dir = report_dir / "test"
# lbl_xtra = ":test:togs"
# mode = "Test"

report_dir.mkdir(exist_ok=True, parents=True)
test_df

,Team ID,Task ID,Run ID,System Description,macro_precision,macro_recall,macro_f1,micro_precision,micro_recall,micro_f1
93,TUGW,T611,hermesneefinetunedloraentitiesnaivebeamnoneher...,CHASTEGw,0.479636,0.749669,0.576056,0.508341,0.835804,0.632184
94,TUGW,T611,hermesneefinetunedloraentitiesnaivebeamnoneher...,CHASTEGw,0.480541,0.749481,0.576177,0.503820,0.830986,0.627308
95,TUGW,T611,hermesneefinetunedloraentitiesnaivebeamnoneher...,CHASTEGw,0.746095,0.422334,0.523402,0.885235,0.508895,0.646270
96,TUGW,T611,hermesneefinetunedloraentitiesnaivebeamnoneher...,CHASTEGw,0.479682,0.748725,0.575219,0.501346,0.828391,0.624651
97,TUGW,T611,hermesneefinetunedloraentitiesnaivebeamnoneher...,CHASTEGw,0.738384,0.427672,0.524773,0.868801,0.512973,0.645071
98,TUGW,T611,hermesneefinetunedloraentitiesnaivebeamnoneher...,CHASTEGw,0.478801,0.748708,0.575115,0.505626,0.832839,0.629236
99,TUGW,T611,hermesneefinetunedloraentitiesnaivebeamnoneher...,CHASTEGw,0.337742,0.455759,0.372688,0.334025,0.537064,0.411882
100,TUGW,T611,hermesneefinetunedloraentitiesnaivebeamnoneher...,CHASTEGw,0.337742,0.455759,0.372688,0.334025,0.537064,0.411882
101,TUGW,T611,naiveentitiesGraphwiseT61113NEREXP8SUB1union,NaiveGw,0.476505,0.754267,0.575251,0.527171,0.841364,0.648201
102,TUGW,T611,naiveentitiesGraphwiseT61124ENSEXP4SUB1union,NaiveGw,0.476359,0.754236,0.574762,0.523898,0.836916,0.644406


In [450]:
eval_results: list[dict] = []


score_map = {
    "macro_precision": "$P$",
    "macro_recall": "$R$",
    "macro_f1": "$F_1$",
    "micro_precision": "$P_{micro}$",
    "micro_recall": "$R_{micro}$",
    "micro_f1": "$F_{1,micro}$",
}


def test_table_to_df(
    table: pd.DataFrame,
    task: str,
) -> pd.DataFrame:
    eval_results = []
    for i, row in table.iterrows():
        run_id: str = row["Run ID"]
        merge_mode = row["Team ID"] == "TUGW"
        if row["Task ID"] != task:
            continue
        eval_result = {f"{k}": v for k, v in row.items() if k in score_map}

        result_dict = extract_flags_from_name(
            run_id, merge_mode=merge_mode, k=None, test_mode=True
        )
        result_dict.update(eval_result)
        eval_results.append(result_dict)
    if len(eval_results) == 0:
        return pd.DataFrame()
    eval_df = pd.DataFrame(eval_results)
    # remove duplicate rows
    eval_df = eval_df.drop_duplicates()
    eval_df.rename(score_map, axis=1, inplace=True)
    valid_cols = [
        c
        for c in [
            "Graphwise",
            "Set",
            "Model",
            "Beams",
            "NED FT",
            "RAG",
            "LoRA",
            "Naive",
            "Filter",
        ]
        if c in eval_df.columns
    ]
    eval_df.set_index(valid_cols, inplace=True)
    eval_df = eval_df.sort_index()
    # if "$F_{1,micro}$" in eval_df.columns:
    #     eval_df = eval_df.sort_values("$F_{1,micro}$")
    return eval_df


task_6_1_1_df = test_table_to_df(test_df, "T611")
task_6_1_2_df = test_table_to_df(test_df, "T612")
task_6_2_1_df = test_table_to_df(test_df, "T621")
task_6_2_2_df = test_table_to_df(test_df, "T622")
task_6_1_1_df

Looking for submission file for hermesneefinetunedloraentitiesnaivebeamnonehermes323BentitiesGraphwiseT61113NEREXP8SUB1union, found staging/TUGW_T612_hermesneefinetunedloraentitiesnaivebeamnonehermes323BentitiesGraphwiseT61113NEREXP8SUB1union_CHASTEGw
Found full run id ['eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities_Graphwise_T611_13_NER-EXP-8-SUB-1_union'] for hermesneefinetunedloraentitiesnaivebeamnonehermes323BentitiesGraphwiseT61113NEREXP8SUB1union
Looking for submission file for hermesneefinetunedloraentitiesnaivebeamnonehermes323BentitiesGraphwiseT61124ENSEXP4SUB1union, found staging/TUGW_T612_hermesneefinetunedloraentitiesnaivebeamnonehermes323BentitiesGraphwiseT61124ENSEXP4SUB1union_CHASTEGw
Found full run id ['eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities_Graphwise_T611_24_ENS-EXP-4-SUB-1_union'] for hermesneefinetunedloraentitiesnaivebeamnonehermes323BentitiesGraphwiseT61124ENSEXP4SUB1union
Looking for submission

$P$  \
Graphwise                                          Set    Model  Beams    NED FT     RAG      LoRA       Naive                  
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.480541   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.476359   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.746095   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.746246   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.479682   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.475719   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.479636   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.476505   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.738384   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.738530   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.478801   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.475611   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.337742   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.308520   

                                                                                                                          $R$  \
Graphwise                                          Set    Model  Beams    NED FT     RAG      LoRA       Naive                  
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.749481   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.754236   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.422334   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.422334   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.748725   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.754395   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.749669   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.754267   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.427672   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.427672   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.748708   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.753100   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.455759   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.460

In [451]:
task_6_1_1_df

$P$  \
Graphwise                                          Set    Model  Beams    NED FT     RAG      LoRA       Naive                  
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.480541   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.476359   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.746095   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.746246   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.479682   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.475719   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.479636   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.476505   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.738384   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.738530   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.478801   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.475611   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.337742   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.308520   

                                                                                                                          $R$  \
Graphwise                                          Set    Model  Beams    NED FT     RAG      LoRA       Naive                  
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.749481   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.754236   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.422334   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.422334   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.748725   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.754395   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.749669   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.754267   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.427672   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.427672   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.748708   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.753100   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.455759   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.460

In [452]:
task_6_1_1_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_1_1.tex",
    # float_format="%.2f",
    caption=f"{mode} Set Result for Task 6.1.1 for various models and approaches.",
    label=f"tab:task:6_1_1{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_1_1_df

$P$  \
Graphwise                                          Set    Model  Beams    NED FT     RAG      LoRA       Naive                  
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.480541   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.476359   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.746095   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.746246   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.479682   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.475719   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.479636   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.476505   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.738384   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.738530   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.478801   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.475611   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.337742   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.308520   

                                                                                                                          $R$  \
Graphwise                                          Set    Model  Beams    NED FT     RAG      LoRA       Naive                  
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.749481   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.754236   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.422334   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.422334   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.748725   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.754395   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.749669   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.754267   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.427672   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.427672   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.748708   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.753100   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.455759   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.460

In [453]:
task_6_1_2_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_1_2.tex",
    # float_format="%.2f",
    caption=f"{mode} Set Result for Task 6.1.2 for various models and approaches.",
    label=f"tab:task:6_1_2{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_1_2_df

$P$  \
Graphwise                                          Set    Model  Beams    NED FT     RAG      LoRA       Naive                  
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.118442   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.113576   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.634085   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.634220   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.266929   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.263035   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.115336   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.111043   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.628193   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.628324   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.270040   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.266644   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.241557   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.218653   

                                                                                                                          $R$  \
Graphwise                                          Set    Model  Beams    NED FT     RAG      LoRA       Naive                  
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.184856   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.179313   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.362610   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.362610   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.422164   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.420767   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.181533   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.176248   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cap$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.367584   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.367584   
                                                   $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.428799   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.426565   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cup$ 3.2 3B $\times$ \checkmark $\times$ \checkmark \checkmark  0.316783   
                                                          Naive  $\times$ $\times$   $\times$ $\times$   \checkmark  0.315

In [454]:
task_6_2_1_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_2_1.tex",
    # float_format="%.2f",
    caption=f"{mode} Set Result for Task 6.2.1 for various models and approaches.",
    label=f"tab:task:6_2_1{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_2_1_df

""


In [455]:
task_6_2_2_df

$P$  \
Graphwise                                          Set    Model  Beams    NED FT   RAG      LoRA       Naive                
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.028328   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.028328   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.028328   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.028328   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cap$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.045630   
                                                   $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.073661   

                                                                                                                      $R$  \
Graphwise                                          Set    Model  Beams    NED FT   RAG      LoRA       Naive                
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.038632   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.038632   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.038632   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.038632   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cap$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.009004   
                                                   $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.204653   

                                                                                                                    $F_1$  \
Graphwise                                          Set    Model  Beams    NED FT   RAG      LoRA       Naive                
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.026640   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.026640   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.026640   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.026640   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cap$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.014492   
                                                   $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.099398   

                                                                                                                 $P_{micro}$  \
Graphwise                                          Set    Model  Beams    NED FT   RAG      LoRA       Naive                   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.060232   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.060232   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.060232   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.060232   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cap$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.341176   
                                                   $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.085140   

                                                                                          

In [456]:
task_6_2_2_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_2_2.tex",
    # float_format="%.2f",
    caption=f"{mode} Set Result for Task 6.2.2 for various models and approaches.",
    label=f"tab:task:6_2_2{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_2_2_df

$P$  \
Graphwise                                          Set    Model  Beams    NED FT   RAG      LoRA       Naive                
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.028328   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.028328   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.028328   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.028328   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cap$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.045630   
                                                   $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.073661   

                                                                                                                      $R$  \
Graphwise                                          Set    Model  Beams    NED FT   RAG      LoRA       Naive                
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.038632   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.038632   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.038632   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.038632   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cap$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.009004   
                                                   $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.204653   

                                                                                                                    $F_1$  \
Graphwise                                          Set    Model  Beams    NED FT   RAG      LoRA       Naive                
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.026640   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.026640   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.026640   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.026640   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cap$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.014492   
                                                   $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$  0.099398   

                                                                                                                 $P_{micro}$  \
Graphwise                                          Set    Model  Beams    NED FT   RAG      LoRA       Naive                   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.060232   
\parbox{2cm}{\vspace*{0.3em}ENS-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.060232   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-1\vspac... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.060232   
\parbox{2cm}{\vspace*{0.3em}NER-EXP-SUB-EL-freq... $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.060232   
\parbox{2cm}{\vspace*{0.3em}gpt-5.shot-gold\vsp... $\cap$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.341176   
                                                   $\cup$ 3.2 3B $\times$ $\times$ $\times$ \checkmark $\times$     0.085140   

                                                                                          

In [457]:
task_6_1_1_df.columns

Index(['$P$', '$R$', '$F_1$', '$P_{micro}$', '$R_{micro}$', '$F_{1,micro}$'], dtype='object')

In [458]:
tasks = {
    "6.1.1": task_6_1_1_df,
    "6.1.2": task_6_1_2_df,
    "6.2.1": task_6_2_1_df,
    "6.2.2": task_6_2_2_df,
}
average_improvements = []

import re

from constrerl.utils import calculate_improvements, df_topk




for task_name, task_df in tasks.items():
    if len(task_df) == 0:
        print(f"No results for Task {task_name}, skipping.")
        continue
    print(f"Processing Task {task_name} with {len(task_df)} results.")
    task_name_formatted = re.sub(r"\.", "_", task_name)

    task_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(
        lambda v: f"{v:.2f}" if isinstance(v, float) else v
    ).to_latex(
        report_dir / f"task_{task_name_formatted}_full.tex",
        caption=f"{mode} Set Results for Task {task_name}",
        label=f"tab:task:{task_name_formatted}{lbl_xtra}:full",
        clines="all;data",
        hrules=True,
    )
    top_k = 10
    task_df_top = df_topk(task_df, top_k)
    task_df_top.style.highlight_max(axis=0, props="textbf:--rwrap;").format(
        lambda v: f"{v:.2f}" if isinstance(v, float) else v
    ).to_latex(
        report_dir / f"task_{task_name_formatted}_top.tex",
        caption=f"Top {top_k} {mode} Set Results for Task {task_name}",
        label=f"tab:task:{task_name_formatted}{lbl_xtra}",
        clines="all;data",
        hrules=True,
    )
    task_df_improved = calculate_improvements(task_df)
    mean_improvements = task_df_improved.mean()

    task_df_improved.style.format(
        lambda v: (
            f"+\\textcolor{{DarkGreen}}{{{v:.2f}}}"
            if v > 0
            else f"\\textcolor{{DarkRed}}{{{v:.2f}}}"
            if isinstance(v, float)
            else v
        )
    ).to_latex(
        report_dir / f"task_{task_name_formatted}_improve.tex",
        caption=f"{mode} Set Improvements for Task {task_name}",
        label=f"tab:task:{task_name_formatted}{lbl_xtra}_improve",
        clines="all;data",
        hrules=True,
    )
    average_improvements.append(mean_improvements.to_dict() | {"Task": task_name})

Processing Task 6.1.1 with 14 results.
Calculating improvements based on keys: ['Beams'] with options: [('Beams', [])] / other keys: ['Graphwise', 'Set', 'Model', 'NED FT', 'RAG', 'LoRA', 'Naive']
No improvements calculated. Returning empty DataFrame.
Processing Task 6.1.2 with 14 results.
Calculating improvements based on keys: ['Beams'] with options: [('Beams', [])] / other keys: ['Graphwise', 'Set', 'Model', 'NED FT', 'RAG', 'LoRA', 'Naive']
No improvements calculated. Returning empty DataFrame.
No results for Task 6.2.1, skipping.
Processing Task 6.2.2 with 6 results.
Calculating improvements based on keys: ['Beams'] with options: [('Beams', [])] / other keys: ['Graphwise', 'Set', 'Model', 'NED FT', 'RAG', 'LoRA', 'Naive']
No improvements calculated. Returning empty DataFrame.


In [459]:
import re

from constrerl.utils import calculate_improvements

pairs = {
    "Task 6.1": [task_6_1_1_df, task_6_1_2_df],
    "Task 6.2": [task_6_2_1_df, task_6_2_2_df],
}
for task_name, (df1, df2) in pairs.items():
    if df1.empty or df2.empty:
        print(f"Skipping {task_name} due to empty DataFrame.")
        continue
    task_name_formatted = re.sub(r"(\.| )", "_", task_name).strip().lower()
    disambiguation_changes = calculate_improvements(df1, df2)
    disambiguation_changes.style.format(
        lambda v: (
            f"+\\textcolor{{DarkGreen}}{{{v:.2f}}}"
            if v > 0
            else f"\\textcolor{{DarkRed}}{{{v:.2f}}}"
            if isinstance(v, float)
            else v
        )
    ).to_latex(
        report_dir / f"{task_name_formatted}_disambiguation.tex",
        caption=f"{mode} Set Disambiguation Changes for {task_name}",
        label=f"tab:task:{task_name_formatted}{lbl_xtra}_disambiguation",
        clines="all;data",
        hrules=True,
    )
disambiguation_changes

No base keys found in DataFrame columns: ['Graphwise', 'Set', 'Model', 'Beams', 'NED FT', 'RAG', 'LoRA', 'Naive', '$P$', '$R$', '$F_1$', '$P_{micro}$', '$R_{micro}$', '$F_{1,micro}$']. Returning original DataFrame.


TypeError: '>' not supported between instances of 'str' and 'int'